# Замена flash-attn на scaled_dot_product_attention (monkey patch)

In [ ]:
import sys, types, importlib.machinery, torch.nn.functional as F

# единый fallback-kernel
def _sdpa(q, k, v, *args, **kw):
    # dropout_p, softmax_scale, causal и прочие kwargs PyTorch игнорирует, но модель их передаёт —
    # поэтому принимаем *args/**kwargs, чтобы вызов не упал.
    return F.scaled_dot_product_attention(q, k, v)

# ✓ пустой sub-модуль bert_padding с заглушками
bp = types.ModuleType("flash_attn.bert_padding")
for _name in ("index_first_axis", "pad_input", "unpad_input"):
    setattr(bp, _name, lambda *a, **k: a[0])

# создаём основной модуль-заглушку
dummy = types.ModuleType("flash_attn")
dummy.__spec__ = importlib.machinery.ModuleSpec("flash_attn", None)

# все известные имена, которые встречаются в проекте
for _n in (
    "flash_attn_func",
    "flash_attn_varlen_func",
    "flash_attn_varlen_qkvpacked_func",
    "flash_attn_varlen_kvpacked_func",
    "flash_attn_qkvpacked_func",
    "flash_attn_kvpacked_func",
):
    setattr(dummy, _n, _sdpa)

# на случай, если появятся другие символы
def __getattr__(name): return _sdpa
dummy.__getattr__ = __getattr__

# регистрируем в sys.modules
sys.modules["flash_attn"] = dummy
sys.modules["flash_attn.bert_padding"] = bp

In [1]:
import sys, types, importlib.machinery, torch.nn.functional as F, pathlib, tempfile

# 1) единый fallback-kernel
def _sdpa(q, k, v, *a, **kw):
    return F.scaled_dot_product_attention(q, k, v)

_sdpa.__module__ = "flash_attn"          # чтобы inspect смотрел в наш модуль

# 2) создаём модуль-заглушку
dummy = types.ModuleType("flash_attn")
dummy.__spec__ = importlib.machinery.ModuleSpec("flash_attn", None)
dummy.__file__ = str(pathlib.Path(tempfile.gettempdir()) / "flash_attn_stub.py")  # любой путь-строка

# 3) все встречаемые имена
names = [
    "flash_attn_func",
    "flash_attn_varlen_func",
    "flash_attn_varlen_qkvpacked_func",
    "flash_attn_varlen_kvpacked_func",
    "flash_attn_qkvpacked_func",
    "flash_attn_kvpacked_func",
]
for n in names:
    setattr(dummy, n, _sdpa)

# 4) заглушка sub-модуля bert_padding
bp = types.ModuleType("flash_attn.bert_padding")
for n in ("index_first_axis", "pad_input", "unpad_input"):
    setattr(bp, n, lambda *a, **k: a[0])

# 5) регистрируем
sys.modules["flash_attn"] = dummy
sys.modules["flash_attn.bert_padding"] = bp

In [ ]:
import sys, types, importlib.machinery, torch, torch.nn.functional as F

# 1) глушим check из transformers: до импорта transformers
import types
import importlib.util

def _no_flash(*a, **k):
    return False                           # всегда говорит: «FlashAttention недоступен»

# Создадим заглушечный пакет flash_attn **без __spec__**,
# чтобы find_spec("flash_attn") вернул None.
flash_stub = types.ModuleType("flash_attn")
flash_stub.flash_attn_func = flash_stub.flash_attn_varlen_func = \
flash_stub.flash_attn_varlen_qkvpacked_func = flash_stub.flash_attn_qkvpacked_func = \
flash_stub.flash_attn_varlen_kvpacked_func = flash_stub.flash_attn_kvpacked_func = \
    lambda q,k,v,*a,**kw: F.scaled_dot_product_attention(q,k,v)

sys.modules["flash_attn"] = flash_stub
sys.modules["flash_attn.bert_padding"] = types.ModuleType("flash_attn.bert_padding")

# 2) когда позже импортнётся transformers, подменим проверку
import builtins
_real_import = builtins.__import__
def _patched_import(name, *args, **kwargs):
    mod = _real_import(name, *args, **kwargs)
    if name == "transformers.utils":
        # 4.34  →  is_flash_attn_available
        # 4.35+ →  is_flash_attn_2_available
        for fn in ("is_flash_attn_available", "is_flash_attn_2_available"):
            if hasattr(mod, fn):
                setattr(mod, fn, _no_flash)
    return mod
builtins.__import__ = _patched_import

In [ ]:
import sys, types, importlib.machinery, torch.nn.functional as F, importlib

# ─────────────────────────────────────────────────────────────
# 1) Заглушка flash_attn с корректным __spec__ и нужными функциями
# ─────────────────────────────────────────────────────────────
def _sdpa(q, k, v, *a, **kw):
    return F.scaled_dot_product_attention(q, k, v)

flash_stub = types.ModuleType("flash_attn")
flash_stub.__spec__ = importlib.machinery.ModuleSpec("flash_attn", loader=None)  # строка-путь не важна
for name in (
    "flash_attn_func",
    "flash_attn_varlen_func",
    "flash_attn_varlen_qkvpacked_func",
    "flash_attn_varlen_kvpacked_func",
    "flash_attn_qkvpacked_func",
    "flash_attn_kvpacked_func",
):
    setattr(flash_stub, name, _sdpa)

bp = types.ModuleType("flash_attn.bert_padding")
for n in ("index_first_axis", "pad_input", "unpad_input"):
    setattr(bp, n, lambda *a, **k: a[0])

sys.modules["flash_attn"] = flash_stub
sys.modules["flash_attn.bert_padding"] = bp

# ─────────────────────────────────────────────────────────────
# 2) Когда загрузится transformers.utils.import_utils,
#    подменяем _is_package_available и все "is_flash_attn_*"
# ─────────────────────────────────────────────────────────────
import builtins
_real_import = builtins.__import__

def _patched_import(name, globals=None, locals=None, fromlist=(), level=0):
    mod = _real_import(name, globals, locals, fromlist, level)
    if name == "transformers.utils.import_utils":          # ← модуль уже загружен
        def _no_flash(*args, **kwargs):
            # всегда говорим, что flash_attn НЕДОСТУПЕН
            if args and args[0] == "flash_attn":
                return (False, "") if kwargs.get("return_version") else False
            # для всех прочих пакетов вызываем оригинальную логику
            return mod._orig_is_package_available(*args, **kwargs)

        if not hasattr(mod, "_orig_is_package_available"):
            mod._orig_is_package_available = mod._is_package_available
            mod._is_package_available = _no_flash

        for fn in ("is_flash_attn_available",
                   "is_flash_attn_2_available",
                   "is_flash_attn_greater_or_equal"):
            if hasattr(mod, fn):
                setattr(mod, fn, lambda *a, **k: False)
    return mod

builtins.__import__ = _patched_import
#####################################################################

# Проверка версий и зависимостей

In [1]:
pip list

Package                   Version
------------------------- ------------------------
accelerate                1.7.0
aiohappyeyeballs          2.6.1
aiohttp                   3.11.18
aiosignal                 1.3.2
antlr4-python3-runtime    4.9.3
anyio                     4.9.0
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.5
async-timeout             5.0.1
attrs                     25.3.0
audioread                 3.0.1
babel                     2.17.0
beautifulsoup4            4.13.4
bleach                    6.2.0
blobfile                  3.0.0
cairocffi                 1.7.1
CairoSVG                  2.8.2
certifi                   2025.4.26
cffi                      1.17.1
charset-normalizer        3.4.2
colorama                  0.4.6
comm                      0.2.2
conformer                 0.3.2
contourpy                 1.3.2
cssselect2                0.8.0
cycler 

In [2]:
import torch, flash_attn
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("flash‑attn:", flash_attn.__version__)

Torch: 2.8.0.dev20250521+cu128 CUDA: 12.8
flash‑attn: 2.7.4.post1


In [3]:
pip list | grep torch

pytorch-lightning         2.5.1.post0
pytorch-triton            3.3.1+gitc8757738
torch                     2.8.0.dev20250521+cu128
torchaudio                2.6.0.dev20250521+cu128
torchcde                  0.2.5
torchdiffeq               0.2.5
torchdyn                  1.0.6
torchmetrics              1.7.1
torchsde                  0.2.6
torchvision               0.22.0.dev20250521+cu128
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
print(torch.cuda.is_available())  # False, если CUDA недоступна
print(torch.cuda.current_device())  # Печатает текущий GPU, если доступен
print(torch.version.cuda)  # Версия CUDA, если она установлена

True
0
12.8


In [9]:
import torch

# Проверка доступности CUDA
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

# Печать версии CUDA (если она доступна)
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("CUDA is not available on this system.")

CUDA is available. Using GPU.
CUDA Version: 12.8


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# Проверка доступности CUDA
print("CUDA доступна:", torch.cuda.is_available())

# Название видеокарты
if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

# Простой тензор и его размещение на GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(3, 3).to(device)
print("Тензор находится на:", x.device)

# Простая модель
model = nn.Sequential(
    nn.Linear(3, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
).to(device)

# Проверим, где находится модель
for name, param in model.named_parameters():
    print(f"{name} находится на: {param.device}")

# Простой шаг обучения
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

target = torch.randn(1).to(device)
output = model(x)
loss = criterion(output, target)

print("Loss:", loss.item())
loss.backward()
optimizer.step()
print("Обучение прошло успешно.")


CUDA доступна: True
Устройство: NVIDIA GeForce RTX 5090
Compute capability: (12, 0)
Тензор находится на: cuda:0
0.weight находится на: cuda:0
0.bias находится на: cuda:0
2.weight находится на: cuda:0
2.bias находится на: cuda:0
Loss: 0.19365142285823822
Обучение прошло успешно.


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [7]:
import torch

# Test PyTorch installation
print("PyTorch version:", torch.__version__)

PyTorch version: 2.8.0.dev20250521+cu128


In [8]:
# Check if CUDA (GPU) is available
if torch.cuda.is_available():
    device = torch.device("cuda")  # Use GPU
    print("CUDA is available! Using GPU.")
else:
    device = torch.device("cpu")  # Use CPU
    print("CUDA is not available. Using CPU.")

# Create a tensor
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor x:\n", x)

# Perform basic operations
y = x + 2
print("Tensor y (x + 2):\n", y)

z = torch.matmul(x, y)
print("Matrix multiplication (x @ y):\n", z)

CUDA is available! Using GPU.
Tensor x:
 tensor([[1., 2.],
        [3., 4.]])
Tensor y (x + 2):
 tensor([[3., 4.],
        [5., 6.]])
Matrix multiplication (x @ y):
 tensor([[13., 16.],
        [29., 36.]])


# Загрузка модели с huggingface

In [10]:
import os
os.environ["HF_HOME"] = "/app/huggingface_cache"

In [11]:
from kimia_infer.api.kimia import KimiAudio
import os
import soundfile as sf




model = KimiAudio(
    model_path="moonshotai/Kimi-Audio-7B-Instruct",
    load_detokenizer=True,
)

sampling_params = {
    "audio_temperature": 0.8,
    "audio_top_k": 10,
    "text_temperature": 0.0,
    "text_top_k": 5,
    "audio_repetition_penalty": 1.0,
    "audio_repetition_window_size": 64,
    "text_repetition_penalty": 1.0,
    "text_repetition_window_size": 16,
}

2025-05-22 09:00:11.126 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model


Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

model.pt:   0%|          | 0.00/19.0G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.17k [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/2.61k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

KeyboardInterrupt: 

# Последний monkey_patch

In [ ]:
# monkey_patch.py
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["FORCE_CUDA"] = "0"  # если хочешь явно отключить CUDA
import types
import torch
import importlib.machinery
import sys

def sdpa(query, key, value, **kwargs):
    return torch.nn.functional.scaled_dot_product_attention(query, key, value, **kwargs)

# Создаём поддельный модуль flash_attn
fake = types.ModuleType("flash_attn")

# Подменяем нужные функции
fake.flash_attn_qkvpacked_func = sdpa
fake.flash_attn_unpadded_func = sdpa
fake.flash_attn_varlen_func = sdpa
fake.flash_attn_varlen_qkvpacked_func = sdpa
fake.flash_attn_func = sdpa

# Добавим __spec__, чтобы importlib не упал
fake.__spec__ = importlib.machinery.ModuleSpec(name="flash_attn", loader=None)

# Дополнительно — заглушки для модулей внутри transformers
fake.index_first_axis = lambda x, y: x
fake.pad_input = lambda x, *a, **kw: x
fake.unpad_input = lambda x, *a, **kw: x

# Регистрируем модуль
sys.modules["flash_attn"] = fake
sys.modules["flash_attn.flash_attn_interface"] = fake
sys.modules["flash_attn.bert_padding"] = fake


# Запуск модели из кэша hugginface на хосте

In [ ]:
#import os
#os.environ["FLASH_ATTENTION_SKIP_CUDA_BUILD"] = "TRUE"
#os.environ["FLASH_ATTENTION_FORCE_BUILD"] = "TRUE"
#---------------------------------------------------------
#import sys, types, importlib.machinery

#dummy = types.ModuleType("flash_attn")
#dummy.__spec__ = importlib.machinery.ModuleSpec("flash_attn", None)  # <-- главное!
#sys.modules["flash_attn"] = dummy
#sys.modules["flash_attn.flash_attn_func"] = dummy          # любые под-модули, которые импортирует модель
#sys.modules["flash_attn.flash_attn_varlen_func"] = dummy
#sys.modules["flash_attn.bert_padding"] = dummy
#---------------------------------------------------------
#import os
os.environ["HF_HOME"] = "/app/huggingface_cache"


import soundfile as sf
# Assuming the KimiAudio class is available after installation
from kimia_infer.api.kimia import KimiAudio
import torch # Ensure torch is imported if needed for device placement

#device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device("cpu")
# Локальный путь к модели (убрали 'snapshots')
model_path = "huggingface_cache/hub/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/a574f67664cb0443ce08fd6827eb7e2170c94140/"

# Загружаем модель из локальной папки
model = KimiAudio(model_path=model_path, load_detokenizer=True)
#model.to(device)  # Пример размещения на устройстве (GPU или CPU)

print("Starting inference...")

# Инференс 

In [ ]:
#model = KimiAudio(
#        model_path="huggingface_cache/hub/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/a574f67664cb0443ce08fd6827eb7e2170c94140/",
#        load_detokenizer=True,
#    )

sampling_params = {
        "audio_temperature": 0.8,
        "audio_top_k": 10,
        "text_temperature": 0.0,
        "text_top_k": 5,
        "audio_repetition_penalty": 1.0,
        "audio_repetition_window_size": 64,
        "text_repetition_penalty": 1.0,
        "text_repetition_window_size": 16,
    }

messages = [
        {"role": "user", "message_type": "text", "content": "please transcribe this audio"},
        {
            "role": "user",
            "message_type": "audio",
            "content": "test_audios/english1.wav",
        },
    ]

wav, text = model.generate(messages, **sampling_params, output_type="text")
print(">>> output text: ", text)

2025-05-19 22:42:09.232 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2025-05-19 22:42:09.233 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in huggingface_cache/hub/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/a574f67664cb0443ce08fd6827eb7e2170c94140/
2025-05-19 22:42:09.234 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2025-05-19 22:42:11.696 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in huggingface_cache/hub/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/a574f67664cb0443ce08fd6827eb7e2170c94140/
2025-05-19 22:42:11.698 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2025-05-19 22:42:14.298 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2025-05-19 22:42:14.636 | INFO     | kimia_infer.api.kimia:__init__:40 - Loading detokenizer
No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'
Detected CUDA files, patching ldflags
Emitting ninja build file /app/kimia_infer/models/detokenizer/vocoder/alias_free_activation/cuda/build/build.ninja...
/usr/local/lib/python3.10/dist-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warni

IndexError: list index out of range

In [ ]:
output_dir = "test_audios/output"
os.makedirs(output_dir, exist_ok=True)
# audio2audio
messages = [
        {
            "role": "user",
            "message_type": "audio",
            "content": "test_audios/english1.wav",
        }
    ]

wav, text = model.generate(messages, **sampling_params, output_type="both")
sf.write(
        os.path.join(output_dir, "output.wav"),
        wav.detach().cpu().view(-1).numpy(),
        24000,
    )
print(">>> output text: ", text))